# Coding Agent

You'll need to build a Coding Agent powered by an LLM that can:
- Clone and explore GitHub repositories
- Read, analyze, and modify code files
- Execute tasks autonomously based on natural language instructions

## Implementación

In [1]:
!pip install openai chromadb "langfuse<3" pyyaml tiktoken requests -q #en las versiones posteriores a 3 no soporta el .trace()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.6/275.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.

### Create OpenAI Client

In [2]:
"""
1
Importación de librerías y API keys
Definición  de modelos
"""

import os
import json
import subprocess
import requests, yaml, re, hashlib
from pathlib import Path
from openai import OpenAI
from datetime import datetime

try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
    langfuse_pk = userdata.get("LANGFUSE_PK")
    langfuse_sk = userdata.get("LANGFUSE_SK")
except Exception:
    api_key = input("Enter your API key: ")
    langfuse_pk = input("Langfuse public key: ")
    langfuse_sk = input("Langfuse secret key: ")

client = OpenAI(
    api_key=api_key
)

MODEL = "gpt-5-nano"
EMBED_MODEL = "text-embedding-3-small"

#creo un directorio de trbajo donde operará el agente
WORKSPACE = Path("/content/workspace")
WORKSPACE.mkdir(parents=True, exist_ok=True)
os.chdir(WORKSPACE)

print("Cliente OpenAI listo")
print(f"Worksapce: {WORKSPACE}")



Cliente OpenAI listo
Worksapce: /content/workspace


### Langfuse

In [3]:
"""
2
Conexión con Langfuse
"""

from langfuse import Langfuse

lf = Langfuse(
    public_key=langfuse_pk,
    secret_key=langfuse_sk,
    host="https://cloud.langfuse.com"
)

try:
    lf.auth_check()
    print("Langfuse conectado")
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")
    print(f"public_key empieza con: {langfuse_pk[:8] if langfuse_pk else 'VACIA'}")
    print(f"secret_key empieza con: {langfuse_sk[:8] if langfuse_sk else 'VACIA'}")

Langfuse conectado


### RAG

In [4]:
"""
5
Descarga documentación de React en memoria
"""

#indexación de documentación React
REACT_DOCS_SOURCES = [
    {
        "url": "https://raw.githubusercontent.com/reactjs/react.dev/main/src/content/learn/index.md",
        "title": "React - Learn"
    },
    {
        "url": "https://raw.githubusercontent.com/reactjs/react.dev/main/src/content/learn/thinking-in-react.md",
        "title": "Thinking in React"
    },
    {
        "url": "https://raw.githubusercontent.com/reactjs/react.dev/main/src/content/learn/passing-props-to-a-component.md",
        "title": "Props"
    },
    {
        "url": "https://raw.githubusercontent.com/reactjs/react.dev/main/src/content/learn/managing-state.md",
        "title": "Managing State"
    },
    {
        "url": "https://raw.githubusercontent.com/reactjs/react.dev/main/src/content/reference/react/hooks.md",
        "title": "Hooks Reference"
    },
    {
        "url": "https://raw.githubusercontent.com/reactjs/react.dev/main/src/content/learn/scaling-up-with-reducer-and-context.md",
        "title": "Reducer and Context"
    },
]

def fetch_doc(url: str) -> str:
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        return r.text
    except Exception as e:
        print(f"  [WARN] No se pudo descargar {url}: {e}")
        return ""

raw_docs = []
for source in REACT_DOCS_SOURCES:
    print(f"Descargando: {source['title']}...")
    content = fetch_doc(source["url"])
    if content:
        raw_docs.append({"title": source["title"], "url": source["url"], "content": content})
        print(f"{len(content)} chars")

print(f"{len(raw_docs)} documentos descargados")

Descargando: React - Learn...
15777 chars
Descargando: Thinking in React...
22965 chars
Descargando: Props...
26905 chars
Descargando: Managing State...
25302 chars
Descargando: Hooks Reference...
5665 chars
Descargando: Reducer and Context...
30942 chars
6 documentos descargados


In [5]:
"""
6
Divide la documentación en chunks
"""

def chunk_text(text: str, title: str, url: str, chunk_size: int = 500, overlap: int = 50) -> list[dict]:
    """Divide el texto en chunks por palabras con overlap."""
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk_words = words[i:i + chunk_size]
        chunk_text = " ".join(chunk_words)
        chunk_id = hashlib.md5(f"{url}:{i}".encode()).hexdigest()
        chunks.append({
            "id": chunk_id,
            "text": chunk_text,
            "title": title,
            "url": url,
            "chunk_index": len(chunks)
        })
        i += chunk_size - overlap
    return chunks

def get_embedding(text: str) -> list[float]:
    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=text[:8000]  # límite de seguridad
    )
    return response.data[0].embedding

# Generar todos los chunks
all_chunks = []
for doc in raw_docs:
    chunks = chunk_text(doc["content"], doc["title"], doc["url"])
    all_chunks.extend(chunks)
    print(f"  {doc['title']}: {len(chunks)} chunks")

print(f"\n✓ Total chunks: {len(all_chunks)}")

  React - Learn: 5 chunks
  Thinking in React: 7 chunks
  Props: 8 chunks
  Managing State: 7 chunks
  Hooks Reference: 2 chunks
  Reducer and Context: 8 chunks

✓ Total chunks: 37


In [6]:
"""
7
Guarda chunks en ChromaDB
RAG search
"""

# ChromaDB - Base vectorial persistente

import chromadb

CHROMA_PATH = WORKSPACE / "chroma_db"

# Si la carpeta ya existe, reutiliza la base.
# Si no existe, la crea automáticamente.
chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_PATH)
)

COLLECTION_NAME = "react_docs"

print(f"✓ ChromaDB inicializado en: {CHROMA_PATH}")

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"description": "React documentation RAG"}
)

if collection.count() == 0:
    print("Indexando chunks...")

    for i, chunk in enumerate(all_chunks):
        embedding = get_embedding(chunk["text"])

        collection.add(
            ids=[str(i)],
            documents=[chunk["text"]],
            embeddings=[embedding],
            metadatas=[{
                "title": chunk["title"],
                "url": chunk["url"]
            }]
        )

        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(all_chunks)} chunks indexados...")

else:
    print(f"✓ Colección existente: {collection.count()} chunks")


print(f"\n✓ RAG listo — {collection.count()} chunks indexados")

# RAG Search usando ChromaDB

def rag_search(query: str, n_results: int = 3) -> list[dict]:

    query_embedding = get_embedding(query)

    response = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )

    results = []

    documents = response["documents"][0]
    metadatas = response["metadatas"][0]

    distances = response.get("distances")

    if distances is None:
        distances = [[0.0] * len(documents)]

    distances = distances[0]

    for document, metadata, distance in zip(
        documents,
        metadatas,
        distances
    ):

        results.append({
            "text": document,
            "title": metadata["title"],
            "url": metadata["url"],
            "score": float(1 - distance),
            "source": "RAG"
        })

    return results

# ==========================================================
# Test
# ==========================================================

print("\nTest RAG:")

resultados = rag_search("React hooks useState useEffect")

for r in resultados:
    print(f"  [{r['source']}] {r['title']} (score: {r['score']:.3f})")
    print(f"    {r['text'][:100]}...")

✓ ChromaDB inicializado en: /content/workspace/chroma_db
Indexando chunks...
  10/37 chunks indexados...
  20/37 chunks indexados...
  30/37 chunks indexados...

✓ RAG listo — 37 chunks indexados

Test RAG:
  [RAG] Props (score: 0.064)
    import { useState, useEffect } from 'react'; import Clock from './Clock.js'; function useTime() { co...
  [RAG] Hooks Reference (score: 0.054)
    a non-reactive event to fire from any Effect hook. --- ## Performance Hooks {/*performance-hooks*/} ...
  [RAG] Managing State (score: 0.048)
    ); } let nextId = 3; ``` ```js src/TaskList.js import { useState, useContext } from 'react'; import ...


### Permisos y espacio de trabajo

In [7]:
"""
3
Definición de permisos dentro del workspace
"""

config_yaml = """
workspace: /content/workspace

permissions:
  read:
    deny:
      - ".env"
      - "**/*.pem"
      - "**/*.key"
      - "secrets/**"
      # - "package.json" #la usamos para la tarea 3
  write:
    deny:
      - ".github/**"
      - "package-lock.json"
      - "yarn.lock"
commands:
  deny:
    - "rm -rf"
    - "git push"
    - "sudo"
    - "chmod"
    - "curl | bash"
    - "wget | bash"
  require_approval:
    - "npm install"
    - "pip install"
    - "git commit"
"""

config_path = WORKSPACE / "agent.config.yaml"
config_path.write_text(config_yaml.strip())
print(f"agent.config.yaml creado en {config_path}")

agent.config.yaml creado en /content/workspace/agent.config.yaml


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Main agent y subagentes

In [8]:
"""
8
Memoria de trabajo, tanto volatil como persistente
Memoria compartida entre subagentes
"""

# ── task_state: estado compartido entre subagentes (vive en memoria por sesión) ──
def new_task_state(request: str, repo_path: str) -> dict:
    return {
        "original_request": request,
        "repo_path": repo_path,
        "progress": [],
        "subagent_results": {
            "explorer": None,
            "researcher": None,
            "implementer": None,
            "tester": None,
            "reviewer": None
        },
        "sources_consulted": [],   # qué docs del RAG o web se usaron
        "files_modified": [],
        "observations": [],
        "rag_chunks_used": [],      # fragmentos recuperados del RAG
        "tool_call_log": []
    }

def log_progress(state: dict, subagent: str, message: str):
    entry = f"[{subagent.upper()}] {message}"
    state["progress"].append(entry)
    print(entry)

#project_memory: persiste entre sesiones en disco
MEMORY_PATH = WORKSPACE / "project_memory.json"

def load_memory() -> dict:
    if MEMORY_PATH.exists():
        with open(MEMORY_PATH) as f:
            memory = json.load(f)
        print(f"✓ Memoria cargada ({len(memory.get('sessions', []))} sesiones previas)")
        return memory
    return {
        "sessions": [],
        "architecture": {}
    }

def save_memory(memory: dict):
    with open(MEMORY_PATH, "w") as f:
        json.dump(memory, f, indent=2, ensure_ascii=False)

def update_memory(memory: dict, state: dict):
    """Al final de cada sesión, actualiza la memoria persistente con lo aprendido."""
    session_summary = {
        "date": datetime.now().isoformat(),
        "request": state["original_request"],
        "repo_path": state["repo_path"],
        "files_modified": state["files_modified"],
        "observations": state["observations"],
        "sources": state["sources_consulted"]
    }
    memory["sessions"].append(session_summary)

    # actualizar arquitectura si el explorer encontró algo
    if state["subagent_results"]["explorer"]:
        memory["architecture"][state["repo_path"]] = state["subagent_results"]["explorer"]

    save_memory(memory)
    print(f"✓ Memoria guardada en {MEMORY_PATH}")

PROJECT_MEMORY = load_memory()
print(f"✓ task_state y project_memory listos")

✓ task_state y project_memory listos


In [9]:
"""
10
Ejecución de subagentes
"""

#subagentes: cada uno usa inner_loop_unificado (con guards activados via state) con su propio system prompt

def _run_subagent(state: dict, supervision: bool, name: str, objetivo: str, extra_context: str = "", span=None) -> str:
    """Arma el mensaje inicial del subagente, corre el loop unificado (con guards, porque se le pasa state) y guarda el resultado en state."""
    system_prompt = (
        f"Sos el subagente '{name}' dentro de un sistema multi-agente de análisis de código.\n"
        f"Tarea original del usuario: {state['original_request']}\n"
        f"Repositorio: {state['repo_path']}\n\n"
        f"Tu objetivo específico:\n{objetivo}\n"
    )
    if extra_context:
        system_prompt += f"\nContexto adicional:\n{extra_context}\n"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": objetivo}
    ]

    log_progress(state, name, "iniciando")
    resultado = inner_loop_unificado(messages, supervision, state=state, span=span, subagent_name=name)
    state["subagent_results"][name] = resultado
    log_progress(state, name, "terminado")
    return resultado


def run_explorer(state: dict, supervision: bool, span=None):
    objetivo = (
        "Explorá la estructura del repositorio con list_files y read_file "
        "(package.json, README, carpetas principales de src/). "
        "Identificá: stack tecnológico, estructura de carpetas, convenciones y componentes clave. "
        "No modifiques ningún archivo. Devolvé un resumen conciso de la arquitectura encontrada."
    )
    return _run_subagent(state, supervision, "explorer", objetivo, span=span)


def run_researcher(state: dict, supervision: bool, span=None):
    query = "React component architecture best practices project structure"
    rag_results = rag_search(query, n_results=3)
    state["rag_chunks_used"].extend(rag_results)
    for r in rag_results:
        state["sources_consulted"].append({"source": "RAG", "title": r["title"], "url": r["url"]})

    rag_context = "\n\n".join(
        f"[{r['title']}] ({r['url']}): {r['text'][:400]}" for r in rag_results
    )

    objetivo = (
        "Investigá buenas prácticas de arquitectura React relevantes al repo analizado. "
        "Usá el contexto de RAG provisto y, si hace falta más información actualizada, "
        "la tool web_search. No modifiques archivos. Devolvé hallazgos clave y recomendaciones."
    )
    resultado = _run_subagent(state, supervision, "researcher", objetivo, extra_context=rag_context, span=span)
    state["sources_consulted"].append({"source": "web_search", "title": "researcher queries", "url": ""})
    return resultado


def run_implementer(state: dict, supervision: bool, span=None):
    explorer_summary = state["subagent_results"].get("explorer") or ""
    researcher_summary = state["subagent_results"].get("researcher") or ""
    report_path = f"{state['repo_path']}/ARCHITECTURE_REPORT.md"

    objetivo = (
        f"Con base en los hallazgos del explorer y el researcher, redactá un reporte de "
        f"arquitectura completo en Markdown y guardalo con write_file en '{report_path}'. "
        "El reporte debe incluir: stack, estructura de carpetas, dependencias, riesgos, comandos relevantes, patrones usados, "
        "y recomendaciones de mejora."
    )
    extra_context = f"Hallazgos explorer:\n{explorer_summary}\n\nHallazgos researcher:\n{researcher_summary}"
    resultado = _run_subagent(state, supervision, "implementer", objetivo, extra_context=extra_context, span=span)

    if Path(report_path).exists() and report_path not in state["files_modified"]:
        state["files_modified"].append(report_path)
    return resultado


def run_tester(state: dict, supervision: bool, span=None):
    report_path = f"{state['repo_path']}/ARCHITECTURE_REPORT.md"
    objetivo = (
        f"Verificá con read_file que '{report_path}' exista y tenga contenido completo "
        "(no vacío, con secciones claras). Si encontrás problemas, indicalos. "
        "No modifiques el archivo, solo reportá el resultado de la verificación."
    )
    return _run_subagent(state, supervision, "tester", objetivo, span=span)


def run_reviewer(state: dict, supervision: bool, span=None):
    explorer_summary = state["subagent_results"].get("explorer") or ""
    researcher_summary = state["subagent_results"].get("researcher") or ""
    implementer_summary = state["subagent_results"].get("implementer") or ""
    tester_summary = state["subagent_results"].get("tester") or ""

    objetivo = (
        "Revisá el trabajo completo del explorer, researcher, implementer y tester. "
        "Dictaminá si el reporte de arquitectura cumple el objetivo original y está listo para entregar. "
        "Empezá tu respuesta con 'APROBADO' o 'RECHAZADO' seguido de una breve justificación."
    )
    extra_context = (
        f"Explorer:\n{explorer_summary}\n\nResearcher:\n{researcher_summary}\n\n"
        f"Implementer:\n{implementer_summary}\n\nTester:\n{tester_summary}"
    )
    return _run_subagent(state, supervision, "reviewer", objetivo, extra_context=extra_context, span=span)


print("✓ Subagentes (explorer, researcher, implementer, tester, reviewer) listos")

✓ Subagentes (explorer, researcher, implementer, tester, reviewer) listos


In [10]:
"""
11
Inicializa el repositorio (una sola vez, al arrancar la sesión)
Ejecuta el pipeline de subagentes sobre un pedido puntual del chat
"""

def init_repo(repo_url: str = None, repo_path: str = None) -> str:
    """
    Clona el repo si se dio URL (o reutiliza el path local si ya existe).
    No crea task_state ni corre subagentes: solo deja el repo listo en disco
    y devuelve el repo_path final. Se llama una única vez, antes del loop de chat.
    """
    if repo_url and not repo_path:
        repo_name = repo_url.rstrip("/").split("/")[-1].replace(".git", "")
        repo_path = str(WORKSPACE / repo_name)

        if Path(repo_path).exists():
            print(f"✓ Repo ya clonado en {repo_path} (memoria)")
        else:
            print(f"Clonando {repo_url}...")
            result = subprocess.run(
                ["git", "clone", "--depth", "1", repo_url, repo_path],
                capture_output=True, text=True
            )
            if result.returncode != 0:
                raise RuntimeError(f"Clone falló:\n{result.stderr}")
            print(f"✓ Repo clonado en {repo_path}")

    if not repo_path or not Path(repo_path).exists():
        raise RuntimeError("Necesitás proveer un repo_url o un repo_path válido.")

    return repo_path


def subagent_pipeline(request: str, repo_path: str, state: dict, trace, supervision: bool = True) -> str:
    """
    Agente principal en modo pipeline: coordina explorer -> researcher -> implementer ->
    tester -> reviewer sobre el pedido actual del turno de chat, reusando el state y el
    trace de la sesión (no los crea, se los pasa run_agent()).
    Devuelve un resumen en texto listo para insertarse en el historial de la conversación.
    """
    log_progress(state, "main", f"Pedido recibido: {request}")

    if repo_path in PROJECT_MEMORY.get("architecture", {}):
        print("[MEMORIA] Ya analicé este repo antes. Usando contexto previo.")

    try:
        span_explorer = trace.span(name="explorer")
        run_explorer(state, supervision, span=span_explorer)
        span_explorer.end(output={"result": str(state["subagent_results"]["explorer"])[:500]})

        span_researcher = trace.span(name="researcher")
        run_researcher(state, supervision, span=span_researcher)
        span_researcher.end(output={
            "result": str(state["subagent_results"]["researcher"])[:500],
            "rag_chunks": len(state["rag_chunks_used"])
        })

        span_implementer = trace.span(name="implementer")
        run_implementer(state, supervision, span=span_implementer)
        span_implementer.end(output={"files_modified": state["files_modified"]})

        span_tester = trace.span(name="tester")
        run_tester(state, supervision, span=span_tester)
        span_tester.end(output={"result": str(state["subagent_results"]["tester"])[:500]})

        span_reviewer = trace.span(name="reviewer")
        run_reviewer(state, supervision, span=span_reviewer)
        span_reviewer.end(output={"result": str(state["subagent_results"]["reviewer"])[:500]})

        update_memory(PROJECT_MEMORY, state)

        resumen = (
            "[PIPELINE COMPLETADO]\n"
            f"Archivos modificados: {state['files_modified']}\n"
            f"Veredicto del reviewer: {state['subagent_results']['reviewer']}"
        )

        trace.update(
            output={"files_modified": state["files_modified"], "status": "completado"},
            metadata={"rag_chunks_used": len(state["rag_chunks_used"])}
        )
        return resumen

    except Exception as e:
        log_progress(state, "main", f"ERROR: {e}")
        trace.update(output={"status": "error", "error": str(e)})
        return f"[PIPELINE ERROR] {e}"

    finally:
        lf.flush()

print("✓ init_repo y subagent_pipeline listos")


✓ init_repo y subagent_pipeline listos


In [11]:
"""
9
Manejo del contexto
Resumen de la conversación
Detecta loops
"""

import tiktoken

TOKENIZER = tiktoken.encoding_for_model("gpt-4o-mini")
MAX_CONTEXT_TOKENS = 6000  # dejamos margen del límite del modelo

def count_tokens(messages: list) -> int:
    """Cuenta tokens aproximados del historial."""
    total = 0
    for m in messages:
        content = m.get("content") or ""
        total += len(TOKENIZER.encode(str(content)))
    return total

def summarize_history(messages: list) -> list:
    """
    Si el historial es muy largo, resume los mensajes del medio
    y conserva el system prompt + los últimos mensajes, respetando
    pares assistant(tool_calls) + tool(response) completos.
    """
    system = [m for m in messages if m["role"] == "system"]
    rest   = [m for m in messages if m["role"] != "system"]

    if len(rest) <= 4:
        return messages

    # Buscar un punto de corte seguro cerca de los últimos 4 mensajes,
    # retrocediendo si hace falta para no partir un par tool_call/tool_response
    cut = len(rest) - 4
    while cut > 0 and rest[cut]["role"] == "tool":
        cut -= 1  # si el mensaje en el corte es un tool_response, retrocedo
                  # hasta encontrar el assistant que lo generó (o antes)

    to_summarize = rest[:cut]
    recent       = rest[cut:]

    if not to_summarize:
        return messages  # nada seguro para resumir, mejor no tocar nada

    history_text = "\n".join(
        f"{m['role'].upper()}: {str(m.get('content', ''))[:300]}"
        for m in to_summarize
    )

    summary_response = client.chat.completions.create(
        model=MODEL,
        messages=[{
            "role": "user",
            "content": f"Resumí en 3-5 oraciones qué se hizo hasta ahora:\n\n{history_text}"
        }]
    )
    summary = summary_response.choices[0].message.content

    summary_msg = {
        "role": "system",
        "content": f"[RESUMEN DE CONTEXTO PREVIO]\n{summary}"
    }

    print(f"  [CONTEXTO] Historial resumido ({len(to_summarize)} mensajes → 1 resumen)")
    return system + [summary_msg] + recent

def make_call_key(subagent_name: str, tool_name: str, args: dict) -> str:
    return f"{subagent_name}:{tool_name}:{json.dumps(args, sort_keys=True)}"

def detect_loop(state: dict, subagent_name: str, tool_name: str, args: dict, threshold: int = 2) -> bool:
    """
    Detecta si el agente está repitiendo la misma acción sin avanzar.
    Retorna True si detecta loop.
    """
    # buscar en el progreso cuántas veces se llamó esta tool con estos mismos args
    key = make_call_key(subagent_name, tool_name, args)
    repetitions = sum(1 for p in state["tool_call_log"] if key in p)

    if repetitions >= threshold:
        print(f"  [LOOP DETECTADO] '{tool_name}' repetido {repetitions+1} veces sin avance.")
        return True
    return False


def inner_loop_unificado(messages: list, supervision: bool, state: dict = None,
                          span=None, subagent_name: str = "main") -> str:
    """
    Loop interno unificado. Reemplaza a inner_loop() e inner_loop_with_guards().

    - Si `state` es None: modo simple, usado por run_agent() para el chat directo
      del agente principal (sin subagente detras). No hay deteccion de loops,
      resumen de contexto ni traza de Langfuse por turno.
    - Si se provee `state`: modo con guards, usado por los subagentes via
      _run_subagent(). Agrega deteccion de loops, resumen de contexto largo,
      logging de progreso en el state compartido y, si se pasa `span`,
      generacion de traza en Langfuse por cada llamada al LLM.
    """
    iteracion = 0

    while True:
        iteracion += 1

        # manejo de contexto largo (solo si hay state, i.e. corrida de subagente)
        if state is not None:
            if count_tokens(messages) > MAX_CONTEXT_TOKENS:
                messages = summarize_history(messages)
            print(f"  [{subagent_name.upper()}] iteración {iteracion}")
        else:
            print(f"Loop interno - iteración: {iteracion}")

        generation = None
        if span is not None:
            generation = span.generation(
                name=f"{subagent_name}-turn-{iteracion}",
                model=MODEL,
                input=messages
            )

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS_SCHEMA,
            tool_choice="auto"
        )

        if generation is not None:
            generation.end(
                output=response.choices[0].message.content,
                usage={
                    "input": response.usage.prompt_tokens,
                    "output": response.usage.completion_tokens,
                    "total": response.usage.total_tokens
                }
            )

        msg = response.choices[0].message

        if not msg.tool_calls:
            messages.append({"role": "assistant", "content": msg.content})
            return msg.content

        messages.append({
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {"name": tc.function.name, "arguments": tc.function.arguments}
                }
                for tc in msg.tool_calls
            ]
        })

        for call in msg.tool_calls:
            tool_name = call.function.name
            tool_args = json.loads(call.function.arguments)

            if state is not None:
                key = make_call_key(subagent_name, tool_name, tool_args)
                state["tool_call_log"].append(key)

                # detección de loop (solo en modo con guards)
                if detect_loop(state, subagent_name, tool_name, tool_args):
                    loop_msg = (
                        f"Detecté que estoy repitiendo '{tool_name}' sin avanzar. "
                        f"Voy a cambiar de estrategia o detenerme."
                    )
                    messages.append({
                        "role": "tool",
                        "tool_call_id": call.id,
                        "content": loop_msg
                    })
                    log_progress(state, subagent_name, f"LOOP DETECTADO en {tool_name} — cambiando estrategia")
                    continue

                log_progress(state, subagent_name, f"tool: {tool_name} {list(tool_args.values())[:1]}")
            else:
                print(f"\nAgente quiere utilizar: {tool_name} " + " ".join(str(v) for v in tool_args.values()))

            result = execute_tool(tool_name, tool_args, supervision)

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": str(result) if result is not None else "Error: la tool no devolvió resultado"
            })

print("✓ Loop unificado listo: modo simple (chat directo) y modo con guards (subagentes)")

✓ Loop unificado listo: modo simple (chat directo) y modo con guards (subagentes)


### Tools

In [12]:
"""
12
Ejecuta tools

Extra opcional (consigna): sistema de plugins para tools.
TOOL_REGISTRY + @register_tool permiten agregar tools nuevas sin tocar
execute_tool, inner_loop_unificado ni el resto del harness: alcanza con
escribir la función, decorarla con @register_tool(...) y llamar a
refresh_tools() para que quede disponible para el agente.
"""

TOOL_REGISTRY: dict = {}

def register_tool(name: str, description: str, parameters: dict, destructive: bool = False):
    """
    Decorador para registrar una tool sin modificar el núcleo del harness.
    - name: nombre que ve el LLM (debe coincidir con el de la función)
    - description: descripción para el schema de function calling
    - parameters: JSON schema de los parámetros (formato OpenAI tools)
    - destructive: si True, requiere aprobación del usuario cuando supervision=True
    """
    def wrapper(func):
        TOOL_REGISTRY[name] = {
            "func": func,
            "destructive": destructive,
            "schema": {
                "type": "function",
                "function": {
                    "name": name,
                    "description": description,
                    "parameters": parameters
                }
            }
        }
        return func
    return wrapper


def refresh_tools():
    """
    Reconstruye TOOLS_SCHEMA, TOOLS_MAP y DESTRUCTIVE_TOOLS a partir de
    TOOL_REGISTRY. Se llama una vez al terminar de registrar las tools base
    (celda siguiente) y de nuevo cada vez que se agrega una tool plugin.
    """
    global TOOLS_SCHEMA, TOOLS_MAP, DESTRUCTIVE_TOOLS
    TOOLS_SCHEMA = [entry["schema"] for entry in TOOL_REGISTRY.values()]
    TOOLS_MAP = {name: entry["func"] for name, entry in TOOL_REGISTRY.items()}
    DESTRUCTIVE_TOOLS = {name for name, entry in TOOL_REGISTRY.items() if entry["destructive"]}
    print(f"✓ Tools registradas: {list(TOOLS_MAP.keys())}")


@register_tool(
    name="read_file",
    description="Lee el contenido completo de un archivo dado su path.",
    parameters={
        "type": "object",
        "properties": {
            "path": {"type": "string", "description": "Path al archivo a leer"}
        },
        "required": ["path"]
    },
    destructive=False,
)
def read_file(path: str, **kwargs) -> str: #--> path --> leo documento
  try:
    with open(path, 'r', encoding='utf-8') as f:
      return f.read()
  except FileNotFoundError:
    return f"Error: el archivo no fue encontrado en '{path}'"
  except Exception as e:
    return f"Error al intentar leer '{path}': {e}"


@register_tool(
    name="write_file",
    description="Escribe (o sobreescribe) contenido en un archivo.",
    parameters={
        "type": "object",
        "properties": {
            "path": {"type": "string", "description": "Path del archivo a escribir"},
            "content": {"type": "string", "description": "Contenido a escribir en el archivo"}
        },
        "required": ["path", "content"]
    },
    destructive=True,
)
def write_file(path: str, content:str) -> str: # escribo contenido en un archivo | reemplazo si ya exite
  try:
    Path(path).parent.mkdir(parents= True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
      f.write(content)
      return f"Archivo escrito con éxito en '{path}'"
  except Exception as e:
    return f"Hubo un error al escribir en '{path}': {e}"


@register_tool(
    name="run_command",
    description="Ejecuta un comando de terminal y devuelve stdout y stderr.",
    parameters={
        "type": "object",
        "properties": {
            "command": {"type": "string", "description": "Comando de terminal a ejecutar"}
        },
        "required": ["command"]
    },
    destructive=True,
)
def run_command(command: str) -> str: # --> ejecuto comando en terminal --> retorno stdout Y stderr
  try:
    result = subprocess.run(
        command,
        shell=True,
        capture_output = True,
        text=True,
        timeout=45
    )
    output = ""
    if result.stdout:
      output += f"\n\nSTDOUT:\n{result.stdout}"
    if result.stderr:
      output += f"\n\nSTDERR:\n{result.stderr}"
    output += f"\nReturn code: {result.returncode}"
    return output if output.strip() else "No hay output"
  except subprocess.TimeoutExpired:
    return "Error: el comando excedió el timeout de 45 segundos"
  except Exception as e:
    return f"Se produjo un error al ejecutar: {e}"


@register_tool(
    name="list_files",
    description="Lista archivos y carpetas en un directorio.",
    parameters={
        "type": "object",
        "properties": {
            "directory": {"type": "string", "description": "Path del directorio a listar (default: '.')"}
        },
        "required": []
    },
    destructive=False,
)
def list_files(directory:str=".") -> str: # listo un directorio, mínimo para que pueda operar
  try:
    p = Path(directory)
    if not p.exists():
      return "El directorio '{directory} especificado no existe"
    items = sorted(p.iterdir())
    lines = [f"Contenido de '{directory}': "]
    for item in items:
      lines.append(f"{item.name}")
    return "\n".join(lines) if len(lines) > 1 else f"'{directory}' está vacío"
  except Exception as e:
    return f"Error listando '{directory}': {e}"


@register_tool(
    name="web_search",
    description="Busca información en la web y devuelve los resultados.",
    parameters={
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Consulta de búsqueda"}
        },
        "required": ["query"]
    },
    destructive=False,
)
def web_search(query:str)->str: # herramienta websearch itnegrada de openai
  try:
    response = client.chat.completions.create(
        model="gpt-5-nano",
        messages=[{"role": "user", "content": query}],
        tools=[{"type": "web_search_preview"}]

    )
    return response.choices[0].message.content
  except Exception as e:
    return f"Error en web_search: {e}"


refresh_tools()

✓ Tools registradas: ['read_file', 'write_file', 'run_command', 'list_files', 'web_search']


In [13]:
"""
13
Define las tools que el agente puede usar

TOOLS_SCHEMA, TOOLS_MAP y DESTRUCTIVE_TOOLS ya NO se escriben a mano acá:
se derivan automáticamente de TOOL_REGISTRY (celda anterior) vía
refresh_tools(). Esta celda solo confirma que quedaron construidas y
define la política de comandos "seguros" que no requieren supervisión.
"""

print(f"TOOLS_SCHEMA: {[t['function']['name'] for t in TOOLS_SCHEMA]}")
print(f"TOOLS_MAP:    {list(TOOLS_MAP.keys())}")
print(f"DESTRUCTIVE_TOOLS: {DESTRUCTIVE_TOOLS}")

# esta es la lista de herramientas que puede ejecutar mi agente. El llm no puede ejecutar código directametne pero si
# pedir ejecutar una función. SIn este esquema, el modelo no puede usar las funciones ya que no sabe cuales existen
# tengo que aclarar cuáles hay, su nombre y los parámetros que speran

import shlex

ALLOWED_COMMANDS = { "ls", "dir", "pwd", "grep", "echo" } # Comandos no destructivos que se debieran permitir sin supervisión
SHELL_OPERATORS = ["|", "||", "&&", ";", "&", ">", ">>", "<", "<<", "$(", "`"]

def is_safe_command(command: str) -> bool:
    try:
        tokens = shlex.split(command)
    except ValueError:
        return False  # comillas mal cerradas u otro problema de parseo -> no confiar

    if not tokens:
        return False

    # si algún token contiene un operador de shell, hay más de un comando
    # encadenado (pipe, subshell, redirección) -> no bypasseamos
    if any(op in tok for tok in tokens for op in SHELL_OPERATORS):
        return False

    binary = tokens[0]
    if binary not in ALLOWED_COMMANDS:
        return False

    if binary == "find" and any(f in tokens for f in ("-delete", "-exec", "-execdir")):
        return False

    return True

TOOLS_SCHEMA: ['read_file', 'write_file', 'run_command', 'list_files', 'web_search']
TOOLS_MAP:    ['read_file', 'write_file', 'run_command', 'list_files', 'web_search']
DESTRUCTIVE_TOOLS: {'write_file', 'run_command'}


### Guardrails

In [14]:
"""
14
Archivo de guardrails
"""

import json

guardrails_config = {
    "allowed_directories": ["/content/workspace"],
    "blocked_paths": ["/etc", "/root"],
    "blocked_commands": ["rm -rf", "git push", "sudo", "chmod"]
}

with open("guardrails.json", "w") as f:
    json.dump(guardrails_config, f, indent=2)

print("guardrails.json creado")

guardrails.json creado


In [15]:
"""
16
Carga de guardrails
"""

def load_guardrails(path="guardrails.json") -> dict:
    try:
        with open(path) as f:
            config = json.load(f)
        print(f"Guardrails cargados: {config}")
        return config
    except FileNotFoundError:
        print("Sin guardrails.json, sin restricciones.")
        return {}

GUARDRAILS = load_guardrails()

Guardrails cargados: {'allowed_directories': ['/content/workspace'], 'blocked_paths': ['/etc', '/root'], 'blocked_commands': ['rm -rf', 'git push', 'sudo', 'chmod']}


In [16]:
"""
4
Validación de toolcalls
Carga los guardrails
"""

import fnmatch # ofrece coincidencia de patrones al estilo de los comodines de terminal Unix

def load_config(path=None) -> dict:
    path = path or WORKSPACE / "agent.config.yaml"
    with open(path, "r") as f:
        return yaml.safe_load(f)

CONFIG = load_config()

def matches_any(value: str, patterns: list) -> bool: #¿existe match entre value y algún pattern de la lista?
    for pattern in patterns:
        if fnmatch.fnmatch(value, pattern):
            return True
    return False

def validate_tool_call(tool_name: str, args: dict) -> str | None:

    # guardrails originales
    if tool_name in ("read_file", "write_file", "list_files"):
        path = args.get("path") or args.get("directory", ".")
        abs_path = str(Path(path).resolve())
        for blocked in GUARDRAILS.get("blocked_paths", []):
            if abs_path.startswith(str(Path(blocked).resolve())):
                return f"Acceso bloqueado a '{path}' por guardrails."
        allowed = GUARDRAILS.get("allowed_directories", [])
        if allowed:
            if not any(abs_path.startswith(str(Path(d).resolve())) for d in allowed):
                return f"'{path}' está fuera de los directorios permitidos."

    # políticas del YAML
    perms = CONFIG.get("permissions", {})
    cmds  = CONFIG.get("commands", {})

    if tool_name == "read_file":
        path = args.get("path", "")
        if matches_any(path, perms.get("read", {}).get("deny", [])):
            return f"[BLOQUEADO] read_file: '{path}' denegado por config."

    if tool_name == "write_file":
        path = args.get("path", "")
        if matches_any(path, perms.get("write", {}).get("deny", [])):
            return f"[BLOQUEADO] write_file: '{path}' denegado por config."

    if tool_name == "run_command":
        command = args.get("command", "")
        for blocked in cmds.get("deny", []):
            if blocked in command:
                return f"[BLOQUEADO] run_command: '{blocked}' es un comando prohibido."
        for needs_ok in cmds.get("require_approval", []):
            if needs_ok in command:
                print(f"\n[APROBACIÓN REQUERIDA] El agente quiere ejecutar: {command}")
                choice = input("¿Permitir? (s/n): ").strip().lower()
                if choice != "s":
                    return f"[CANCELADO] El usuario rechazó ejecutar: {command}"

    return None

print("Configuración cargada:")
print(f"  - read deny:  {CONFIG['permissions']['read']['deny']}")
print(f"  - write deny: {CONFIG['permissions']['write']['deny']}")
print(f"  - cmd deny:   {CONFIG['commands']['deny']}")
print(f"  - approval:   {CONFIG['commands']['require_approval']}")

Configuración cargada:
  - read deny:  ['.env', '**/*.pem', '**/*.key', 'secrets/**', 'package.json']
  - write deny: ['.github/**', 'package-lock.json', 'yarn.lock']
  - cmd deny:   ['rm -rf', 'git push', 'sudo', 'chmod', 'curl | bash', 'wget | bash']
  - approval:   ['npm install', 'pip install', 'git commit']


### Documentación de arquitectura (entregable 4 — generada por código)

In [17]:
"""
Genera ARQUITECTURA.md en el workspace: documenta el rol del agente
principal, el rol de cada subagente y la estructura del estado compartido
(entregable 4 de la consigna: "Breve explicación de la arquitectura").

Esto documenta el DISEÑO del sistema (fijo, no depende de la corrida),
por eso se arma como un string y se escribe a disco apenas se define,
sin depender de que se haya corrido run_agent() todavía.
"""

ARCHITECTURE_DOC = """# Arquitectura del sistema

## 1. Agente principal (`run_agent`)

Es el punto de entrada conversacional. Corre en un loop de chat (`while True`
esperando `input()`), mantiene su propio historial de mensajes (`messages`)
y decide, turno a turno, si:

- responde directo en modo chat (`inner_loop_unificado(messages, supervision)`,
  sin subagentes ni guards), o
- delega el pedido al pipeline de subagentes (`subagent_pipeline(...)`) cuando
  el usuario escribe `/analyze`.

Antes de arrancar el chat, `init_repo()` clona (o reutiliza) el repositorio una
única vez. El agente principal crea y reutiliza durante toda la sesión:
- un `state` (`new_task_state`) compartido con los subagentes,
- un `trace` de Langfuse (`lf.trace(...)`) donde cuelgan los spans de cada
  subagente.

Comandos que expone: `/help`, `/analyze`, `/plan`, `/supervision`, `/config`,
`/status`, `/reset`, `/exit`.

## 2. Subagentes (orquestados por `subagent_pipeline`)

Cada subagente corre `inner_loop_unificado` con guards activados (porque se
le pasa `state`) y su propio `span` de Langfuse. `subagent_pipeline` los
ejecuta siempre en este orden: explorer → researcher → implementer → tester
→ reviewer.

| Subagente | Responsabilidad | Función |
|---|---|---|
| Explorer | Recorre el repo con `list_files`/`read_file`, identifica stack, estructura de carpetas y componentes clave. No modifica archivos. | `run_explorer` |
| Researcher | Consulta el RAG (`rag_search`) sobre buenas prácticas de arquitectura React y, si hace falta, `web_search`. Registra fuentes en `state["sources_consulted"]`. | `run_researcher` |
| Implementer | Con los hallazgos de explorer + researcher, redacta `ARCHITECTURE_REPORT.md` del repo analizado y lo guarda con `write_file`. | `run_implementer` |
| Tester | Verifica con `read_file` que el reporte exista y tenga contenido; no lo modifica. | `run_tester` |
| Reviewer | Revisa el trabajo de los cuatro anteriores y dictamina `APROBADO`/`RECHAZADO` con justificación. | `run_reviewer` |

No todos comparten las mismas tools en la práctica: explorer/tester solo
leen, researcher agrega RAG/web, implementer es el único que escribe el
reporte. Todos pasan igual por `validate_tool_call` (políticas del YAML).

## 3. Estado compartido (`task_state`, ver `new_task_state`)

Vive en memoria durante la sesión y se pasa por referencia a cada subagente:

- `original_request`: pedido original del usuario.
- `repo_path`: repo sobre el que se está trabajando.
- `progress`: lista de eventos (`log_progress`), incluye inicio/fin de cada
  subagente y detecciones de loop.
- `subagent_results`: output de texto de cada subagente (explorer, researcher,
  implementer, tester, reviewer).
- `sources_consulted`: qué se consultó y de dónde (RAG, web_search),
  diferenciado de lo que viene del repo (tools `read_file`/`list_files`,
  visible en `progress`) o de la memoria persistente (aviso `[MEMORIA]`).
- `rag_chunks_used`: fragmentos concretos recuperados del RAG.
- `files_modified`: archivos escritos (ej. `ARCHITECTURE_REPORT.md`).
- `observations`: notas relevantes.
- `tool_call_log`: historial de llamadas a tools, insumo de `detect_loop`.

## 4. Memoria persistente entre sesiones (`project_memory.json`)

Se carga con `load_memory()` al iniciar y se actualiza con `update_memory()`
al terminar cada corrida del pipeline: guarda un resumen por sesión
(`sessions`) y la arquitectura detectada por repo (`architecture`), para que
`subagent_pipeline` avise `[MEMORIA] Ya analicé este repo antes` si vuelve a
verlo.

## 5. RAG

Documentación de React descargada (`REACT_DOCS_SOURCES`) → chunking por
palabras con overlap (`chunk_text`) → embeddings (`text-embedding-3-small`,
`get_embedding`) → almacenamiento vectorial persistente en ChromaDB
(`chroma_client.PersistentClient`, colección `react_docs`). `rag_search`
devuelve texto, título, url y score por chunk recuperado. El researcher
consulta el RAG primero y usa `web_search` como fallback/complemento.

## 6. Config y guardrails (`agent.config.yaml`, `guardrails.json`)

`validate_tool_call` corre en cada llamada a tool y valida, en este orden:
guardrails de paths (`blocked_paths`, `allowed_directories`), políticas de
lectura/escritura del YAML (`permissions.read.deny` / `write.deny`) y
políticas de comandos (`commands.deny` bloquea, `commands.require_approval`
pide confirmación por `input()`).

## 7. Manejo de contexto y loops

`inner_loop_unificado` resume el historial (`summarize_history`) cuando
supera `MAX_CONTEXT_TOKENS`, solo en modo subagente (`state is not None`).
`detect_loop` compara la clave `(subagente, tool, args)` contra
`state["tool_call_log"]`; si se repite `threshold` veces, corta la tool call,
lo deja registrado en `progress` como `LOOP DETECTADO ... — cambiando
estrategia` y el subagente sigue sin ejecutar esa acción de nuevo.

## 8. Observabilidad (Langfuse)

Un `trace` por sesión de chat (`react-architecture-agent-chat`), un `span`
por subagente dentro de `subagent_pipeline`, y una `generation` por llamada
al LLM dentro de `inner_loop_unificado` (prompt, modelo, tokens, latencia)
cuando se le pasa el `span` correspondiente.

## 9. Plugins (extra opcional)

`TOOL_REGISTRY` + `@register_tool(...)` permiten agregar tools nuevas
(ver `find_todos` como ejemplo) sin tocar `execute_tool` ni
`inner_loop_unificado`: alcanza con decorar la función y llamar a
`refresh_tools()` para reconstruir `TOOLS_SCHEMA`/`TOOLS_MAP`/`DESTRUCTIVE_TOOLS`.
"""

def generate_architecture_doc(path=None) -> str:
    path = str(path or (WORKSPACE / "ARQUITECTURA.md"))
    with open(path, "w", encoding="utf-8") as f:
        f.write(ARCHITECTURE_DOC.strip() + "\n")
    print(f"✓ ARQUITECTURA.md generado en {path}")
    return path

generate_architecture_doc()


✓ ARQUITECTURA.md generado en /content/workspace/ARQUITECTURA.md


'/content/workspace/ARQUITECTURA.md'

### Loops

In [18]:
PROMPT = """Sos un agente de código cuyo trabajo es ayudar al usaurio con sus tareas de código dentro del framework de React.
Podes usar las herramietnas disponibles, estas son: read_file, write_file, run_command, list_files y web_search.
Respetá los siguietnes pasos al recibir una tarea:
1) Analizá los requisitos y pasos necesarios para resolver el problema
2) Hacé uso de las tools, de forma iterativa, para cumplir los objetivos
3) Verificá que el trabajo hecho sea correcto con, por ejemplo, tests
4) Reportá el resultado al usuario y explicá cómo lo resolviste

Además de mapear stack y estructura, identificá y leé los archivos que manejan:
- autenticación / sesión / logout
- lectura de localStorage o sessionStorage
- manejo de fechas y duraciones

Para cada uno, evaluá si hay riesgos concretos: invalidación de caché incompleta,
lecturas no reactivas de estado en React, o fragilidad en parsing/cálculo de fechas.

Mantené al usuario siempre al tanto de qué y por qué hacés lo que hacés."""


def execute_tool(name: str, args: dict, supervision: bool) -> str: # dict --> []
  args = {k.strip().rstrip('?'): v for k, v in args.items()}

  # tool inexistente -> error como resultado, no KeyError
  if name not in TOOLS_MAP:
      return f"Error: la tool '{name}' no existe. Tools disponibles: {list(TOOLS_MAP.keys())}"

  error = validate_tool_call(name, args) # valido guardrails
  if error:
      print(error)
      return error  # el LLM se entera y busca otra forma

  # antes: (supervision and name == "run_command") is is_safe_command(...)
  # esa comparacion con "is" devolvia True por accidente para tools sin "command"
  # en sus args (ej write_file), bypaseando la supervision para CUALQUIER tool
  # destructiva que no fuera run_command. Ahora allow_command solo puede ser True
  # para run_command con un comando explicitamente seguro; el resto de las tools
  # destructivas (write_file, plugins destructivos futuros) siempre respetan supervision.
  allow_command = name == "run_command" and is_safe_command(args.get("command", ""))

  if supervision and name in DESTRUCTIVE_TOOLS and not allow_command:
    message = name
    if name == "run_command":
      message += " " + " ".join(str(v) for v in args.values())
    print(f"\n[SUPERVISIÓN] El agente quiere ejecutar: {message}")
    choice = input("¿Permitir? (s/n): ").strip().lower() # strip elimina por defecto los espacios en blanco, tabulaciones y saltos de línea
    if choice != 's':
      return f"Cancelando {name}..."

  # cualquier excepción al ejecutar la tool se convierte en string de error
  # en vez de propagarse (esto dejaba tool_calls sin respuesta y causaba
  # el BadRequestError al mandar el historial a plan_mode_flow)
  try:
      func = TOOLS_MAP[name]
      result = func(**args) #** desempaqueta el diccionario
  except Exception as e:
      result = f"Error ejecutando la tool '{name}' con args {args}: {e}"

  return result

# inner_loop() unificado con inner_loop_with_guards() -> ver inner_loop_unificado() en la celda de guards/contexto


def sanitize_messages(messages: list) -> list:
    """
    Red de seguridad extra: elimina cualquier mensaje 'assistant' con
    tool_calls que no tenga TODAS sus respuestas 'tool' correspondientes
    inmediatamente después. Evita mandar a la API un historial inválido,
    que es lo que causaba el BadRequestError al llamar a plan_mode_flow.
    """
    clean = []
    i = 0
    while i < len(messages):
        m = messages[i]
        if m.get("role") == "assistant" and m.get("tool_calls"):
            expected_ids = {tc["id"] for tc in m["tool_calls"]}
            j = i + 1
            found_ids = set()
            while j < len(messages) and messages[j].get("role") == "tool":
                found_ids.add(messages[j].get("tool_call_id"))
                j += 1
            if expected_ids.issubset(found_ids):
                clean.extend(messages[i:j])
            i = j
        else:
            clean.append(m)
            i += 1
    return clean


def plan_mode_flow(user_message: str, messages: list) -> bool: # armo plan y espero confirmación del usuario
  print("\n[PLAN] Generando plan...")
  safe_history = sanitize_messages(messages)
  plan_messages = safe_history + [{
      "role": "user",
      "content": (
          f"Tarea: {user_message}\n\n"
          "Antes de hacer cualquier acción, describí detalladamente el plan de pasos "
          "que seguirías para completar esta tarea. No ejecutes ninguna tool todavía, "
          "solo listá los pasos."
      ) # agrego prompt al historial
  }]

  plan_response = client.chat.completions.create(
      model=MODEL,
      messages=plan_messages,
  )
  plan = plan_response.choices[0].message.content
  print(f"Plan propuesto:\n{plan}")

  choice = input("\n¿Aprobás este plan? (s/n/modificar): ").strip().lower()
  if choice == 'n':
      print("Tarea cancelada.")
      return False
  elif choice == 'modificar':
      modification = input("Describí los cambios al plan: ").strip() # prompt de modificación
      messages[-1]["content"] += f"\n\nModificación al plan: {modification}"
  return True


REPO_URL = "https://github.com/emmabeni27/pickndrive-ia"


def run_agent(): # loop externo, chat interactua con agente. Comandos: plan, supervision, subagents, reset, exit
  messages = [{"role": "system", "content": PROMPT}]
  plan_mode = True # prendido por defecto
  supervision = True
  subagents_mode = False # pipeline de subagentes apagado por defecto

  print("Inicializando repositorio...")
  try:
      repo_path = init_repo(repo_url=REPO_URL)
  except Exception as e:
      print(f"[ERROR] No se pudo inicializar el repositorio: {e}")
      return

  # state y trace de sesion: se crean una sola vez y los reutiliza cada corrida del pipeline
  state = new_task_state("Análisis conversacional del repositorio", repo_path)
  trace = lf.trace(
      name="react-architecture-agent-chat",
      input={"repo_path": repo_path},
      metadata={"model": MODEL}
  )

  help_string = ("Comandos:\n" +
  "/help, /commands: muestra este texto\n" +
  "/analyze: ejecuta el análisis del repositorio\n" +
  "/plan: des/activa el paso a paso/n" +
  "/supervision: des/activa control sobre operaciones críticas y verbosidad de subagentes\n" +
  # "/subagents: des/activa el pipeline de análisis con subagentes\n" +
  "/config, /status: muestra la configuración de variables actual\n" +
  "/reset: borra el historial\n" +
  "/exit: abandonar chat)")

  print("Agente listo")
  print("="*50)
  print(help_string)
  #print(f"Estado inicial → Plan mode: {'ON' if plan_mode else 'OFF'} | Supervisión: {'ON' if supervision else 'OFF'} | Subagentes: {'ON' if subagents_mode else 'OFF'}")
  print(f"Estado inicial → Plan mode: {'ON' if plan_mode else 'OFF'} | Supervisión: {'ON' if supervision else 'OFF'}")
  print(f"Repositorio activo: {repo_path}")
  print("="*50)

  while True: # loop ext espera input de user
      try:
          user_input = input("\nPrompt: ").strip()
      except (KeyboardInterrupt, EOFError):
          print("\nError. Saliendo...")
          break

      if not user_input:
          continue

      analyze = False

      # Comandos especiales
      if user_input == "/exit":
          print("\n¡Hasta luego!")
          break
      elif user_input == "/reset":
          messages = [{"role": "system", "content": PROMPT}]
          print("\nHistorial reseteado.")
          continue
      elif user_input == "/plan":
          plan_mode = not plan_mode
          print(f"\n> Plan mode: {'ON' if plan_mode else 'OFF'}")
          continue
      elif user_input == "/supervision":
          supervision = not supervision
          print(f"\n> Supervisión: {'ON' if supervision else 'OFF'}")
          continue
      # DESACTIVADO
      elif user_input == "/subagents" and False:
          subagents_mode = not subagents_mode
          print(f"\n> Pipeline de subagentes: {'ON' if subagents_mode else 'OFF'}")
          continue
      elif user_input == "/config" or user_input == "/status":
          print(f"\nConfiguración actual:")
          print(f"> Plan mode: {'ON' if plan_mode else 'OFF'}")
          print(f"> Supervisión: {'ON' if supervision else 'OFF'}")
          #print(f"> Pipeline de subagentes: {'ON' if subagents_mode else 'OFF'}")
          continue
      elif user_input == "/help" or user_input == "/commands":
          print("\n" + help_string)
          continue
      elif user_input == "/analyze":
          analyze = True

      messages.append({"role": "user", "content": user_input}) # msg de user al hisotiral

      # muestro plan --> pido aprobación (aplica tanto si el turno va al pipeline como si va al chat directo)
      if plan_mode:
          approved = plan_mode_flow(user_input, messages[:-1])
          if not approved:
              messages.pop()  # saco el mensaje del usuario si se canceló
              continue

      print("\nAgente: ", end="", flush=True)
      try:
          if subagents_mode or analyze:
              # el pedido aprobado se enruta al pipeline de subagentes,
              # reusando el state y el trace de la sesión
              response = subagent_pipeline(
                  request=user_input,
                  repo_path=repo_path,
                  state=state,
                  trace=trace,
                  supervision=supervision
              )
              messages.append({"role": "assistant", "content": response})
          else:
              # chat directo: inner_loop_unificado ya appendea la respuesta a messages
              response = inner_loop_unificado(messages, supervision)
          print(f"\nAgente: {response}\n")
      except Exception as e:
          print(f"\nError en el agente: {e}\n")

      print("-"*50)


### Funciones utilitarias

In [19]:
"""
15
Funciones para tests
"""

# Funciones para crear archivos restringidos -> el agente no debiera poder acceder o modificarlos

# Archivo con permiso para todos -> el agente no debiera poder hacer chmod
# Intentar chmod 400 test.txt
def create_file_with_full_access(file_name, content):
    with open(file_name, "w", encoding="utf-8") as f:
        f.write(content)
    os.chmod(file_name, 0o777)
    print(f"Archivo {file_name} creado con éxito y permisos totales.")

create_file_with_full_access("test.txt", "You shouldn't be able to chmod this!")

# Crear un directorio restringido
# Intentar que acceda
def crear_directorio(nombre_carpeta):
    try:
        # Crea la carpeta.
        # parents=True crea carpetas intermedias si no existen.
        # exist_ok=True evita errores si ya existe.
        os.makedirs(nombre_carpeta, exist_ok=True)
        print(f"Directorio '{nombre_carpeta}' listo.")
    except Exception as e:
        print(f"Error al crear directorio: {e}")

crear_directorio("root")

Archivo test.txt creado con éxito y permisos totales.
Directorio 'root' listo.


### Plugins — ejemplo de tool nueva sin tocar el núcleo

Demostración del sistema de plugins (extra opcional de la consigna): se agrega una tool
nueva (`find_todos`) sin modificar `execute_tool`, `inner_loop_unificado`, `TOOLS_SCHEMA`
ni `TOOLS_MAP` a mano. Alcanza con escribir la función, decorarla con `@register_tool` y
llamar a `refresh_tools()`.

In [20]:
"""
17
Plugin de ejemplo: nueva tool registrada en runtime
"""

@register_tool(
    name="find_todos",
    description=(
        "Busca comentarios TODO, FIXME o HACK en los archivos de un directorio "
        "del repositorio (recursivo) y devuelve archivo:linea:texto de cada uno."
    ),
    parameters={
        "type": "object",
        "properties": {
            "directory": {
                "type": "string",
                "description": "Directorio donde buscar (default: workspace actual)"
            }
        },
        "required": []
    },
    destructive=False,  # es de solo lectura, no necesita supervision
)
def find_todos(directory: str = ".") -> str:
    try:
        matches = []
        pattern = re.compile(r"(TODO|FIXME|HACK)", re.IGNORECASE)
        extensiones = {".js", ".jsx", ".ts", ".tsx", ".md", ".py"}
        for path in Path(directory).rglob("*"):
            if path.is_file() and path.suffix in extensiones:
                try:
                    with open(path, "r", encoding="utf-8", errors="ignore") as f:
                        for i, line in enumerate(f, start=1):
                            if pattern.search(line):
                                matches.append(f"{path}:{i}: {line.strip()}")
                except Exception:
                    continue
        if not matches:
            return "No se encontraron TODO/FIXME/HACK en el directorio."
        return "\n".join(matches[:50])  # tope de 50 resultados
    except Exception as e:
        return f"Error buscando TODOs en '{directory}': {e}"


# El plugin queda disponible para el agente sin tocar el nucleo del harness:
refresh_tools()

✓ Tools registradas: ['read_file', 'write_file', 'run_command', 'list_files', 'web_search', 'find_todos']


#Notas (Borrar)

## El agente:
* Clona el repositorio
* Puede generar un resumen de la ejecución y da una justificación breve ejecutando todos los subagentes
* Mantiene la conversación

## Cambios:
* Se cambió run_main_agent() por run_agent()
* Se refactorizaron los prints
* Se agregó una lista de comandos no destructivos para que el agente no deba pedir permiso para usarlos
* Agregadas acciones /help, /commands para obtener la lista de comandos, /config, /status para ver la configuración de variables, /analyze para iniciar el análisis del repositorio
* El análizis se inicia manualmente por el usuario con /analize, el agente no tiene acceso a dicha función pero sí a su output

## Por hacer:
* Integrar el análisis como tool para que el agente pueda decidir cuándo analizar si es conveniente

## Ejecución

In [21]:
run_agent()

Inicializando repositorio...
Clonando https://github.com/emmabeni27/pickndrive-ia...
✓ Repo clonado en /content/workspace/pickndrive-ia
Agente listo
Comandos:
/help, /commands: muestra este texto
/analyze: ejecuta el análisis del repositorio
/plan: des/activa el paso a paso/n/supervision: des/activa control sobre operaciones críticas y verbosidad de subagentes
/config, /status: muestra la configuración de variables actual
/reset: borra el historial
/exit: abandonar chat)
Estado inicial → Plan mode: ON | Supervisión: ON
Repositorio activo: /content/workspace/pickndrive-ia

Prompt: /analyze

[PLAN] Generando plan...
Plan propuesto:
Plan detallado de pasos para completar la tarea "/analyze"

Objetivo
- Generar un informe claro sobre el estado del código relacionado con autenticación/sesión, uso de localStorage/sessionStorage y manejo de fechas/duraciones dentro del proyecto React, identificar riesgos (invalidación de caché, lecturas no reactivas, parsing de fechas) y proponer mejoras. No 

In [22]:
import json
mem = json.load(open("/content/workspace/project_memory.json"))
print(f"Sesiones guardadas: {len(mem['sessions'])}")
for s in mem["sessions"]:
    print(s["date"], "-", s["request"])

Sesiones guardadas: 1
2026-07-12T04:17:14.094823 - Análisis conversacional del repositorio
